In [1]:
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from scipy.special import logsumexp
from scipy.optimize import minimize_scalar
from scipy.linalg import sqrtm
import IPython.display as display_lib

## 1D Exact Sampler & Intensity Shift

In [2]:
np.random.seed(42)  # For reproducibility

# ==============================================================================
# 1. Analytical Environments (Multi-SDE Setup)
# We define three distinct bounded scalar potentials A(x) to test varying 
# dynamics: multimodality, robust saturating gradients, and metastability.
# ==============================================================================

class AnalyticalSDE:
    def __init__(self, name, A_func, drift_func, ddrift_func):
        self.name = name
        self.A = A_func
        self.drift = drift_func
        self.ddrift = ddrift_func
        
        # Rigorous bound computation for the stationary potential
        res_A_max = minimize_scalar(lambda x: -self.A(x), bounds=(-10, 10), method='bounded')
        self.A_max = self.A(res_A_max.x)
        
        def phi(x): return 0.5 * self.drift(x)**2 + 0.5 * self.ddrift(x)
        res_phi_min = minimize_scalar(lambda x: phi(x), bounds=(-10, 10), method='bounded')
        res_phi_max = minimize_scalar(lambda x: -phi(x), bounds=(-10, 10), method='bounded')
        
        # [METHODOLOGICAL FIX] Documenting the Girsanov shift (phi_{min}):
        # Factoring out exp(-phi_{min} * tau) modifies the global normalizing constant 
        # but leaves the exact path measure invariant. This mathematically guarantees 
        # \tilde{phi} >= 0 for Poisson thinning (satisfying Assumption 2.3).
        self.phi_min = phi(res_phi_min.x)
        self.M_intensity = phi(res_phi_max.x) - self.phi_min
        
    def phi_tilde(self, x):
        """Mathematically valid shifted intensity: \tilde{phi}(x) = phi(x) - phi_min."""
        return 0.5 * self.drift(x)**2 + 0.5 * self.ddrift(x) - self.phi_min

sde1 = AnalyticalSDE("Periodic", lambda x: np.cos(x), lambda x: -np.sin(x), lambda x: -np.cos(x))
sde2 = AnalyticalSDE("Sigmoidal Proxy", lambda x: -2*np.log(np.cosh(x)), lambda x: -2*np.tanh(x), lambda x: -2/np.cosh(x)**2)
sde3 = AnalyticalSDE("Asymmetric Periodic", lambda x: np.sin(x) + 0.5*np.sin(2*x), lambda x: np.cos(x) + np.cos(2*x), lambda x: -np.sin(x) - 2*np.sin(2*x))

In [3]:
# ==============================================================================
# 2. Simulation Solvers
# ==============================================================================

def euler_maruyama(sde, y0, N_samples, dt, T):
    """Standard Euler-Maruyama acting on the Unit-Volatility Forward Equation."""
    np.random.seed(42)
    y = np.full(N_samples, float(y0))
    n_steps = int(np.ceil(T / dt))
    nfe = 0
    for _ in range(n_steps):
        y += sde.drift(y) * dt + np.sqrt(dt) * np.random.randn(N_samples)
        nfe += 1
    return y, nfe

def segs_exact_sampler(sde, y0, N_samples, dtau, T):
    """
    Corrected Sequential Exact Generative Sampling (SEGS).
    Enforces strict temporal covariance via sequential Brownian bridge updates.
    """
    np.random.seed(42)
    y = np.full(N_samples, float(y0))
    n_steps = int(np.ceil(T / dtau))
    total_nfe = 0
    
    for step in range(n_steps):
        dt_curr = min(dtau, T - step * dtau)
        active_idx = np.arange(N_samples)
        y_next = np.zeros(N_samples)
        
        while len(active_idx) > 0:
            K_active = len(active_idx)
            
            # --- PHASE 1 & 2: Gaussian Endpoints & Reweighting ---
            y_cand = y[active_idx] + np.sqrt(dt_curr) * np.random.randn(K_active)
            total_nfe += K_active 
            
            prob_end = np.exp(sde.A(y_cand) - sde.A_max)
            end_mask = np.random.rand(K_active) <= prob_end
            
            idx_pass = active_idx[end_mask]
            start_pass = y[idx_pass]
            cand_pass = y_cand[end_mask]
            K_pass = len(idx_pass)
            if K_pass == 0: continue
                
            # --- PHASE 3: Sequential Poisson Stochastic Thinning ---
            K_arrivals = np.random.poisson(sde.M_intensity * dt_curr, size=K_pass)
            path_accepted = np.ones(K_pass, dtype=bool)
            
            for j in range(K_pass):
                k_arr = K_arrivals[j]
                if k_arr == 0: continue
                    
                # Sort arrivals to enforce sequential conditioning
                s_times = np.sort(np.random.uniform(0, dt_curr, k_arr))
                curr_y, curr_t = start_pass[j], 0.0
                y_end = cand_pass[j]
                reject = False
                
                # Simulate true Brownian bridge temporal covariance sequentially (Eqs. 48-49)
                for k in range(k_arr):
                    t_next = s_times[k]
                    dt_rem = dt_curr - curr_t
                    
                    mean_next = curr_y + (t_next - curr_t) / dt_rem * (y_end - curr_y)
                    var_next = (t_next - curr_t) * (dt_curr - t_next) / dt_rem
                    next_y = mean_next + np.sqrt(max(var_next, 1e-12)) * np.random.randn()
                    
                    curr_y, curr_t = next_y, t_next
                    total_nfe += 1
                    
                    # Evaluate shifted intensity on the mathematically valid trajectory
                    intensity = sde.phi_tilde(next_y)
                    if np.random.rand() <= intensity / sde.M_intensity:
                        reject = True
                        break  # Path measure rejected; short-circuit remaining sequence
                        
                if reject: path_accepted[j] = False
                    
            final_idx = idx_pass[path_accepted]
            y_next[final_idx] = cand_pass[path_accepted]
            active_idx = active_idx[~np.isin(active_idx, final_idx)]
            
        y = y_next
    return y, total_nfe / N_samples

In [4]:
# ==============================================================================
# 3. Execution & Metrics Collection
# ==============================================================================
N_SAMPLES, T_END, Y0 = 20000, 2.0, 0.0
results = []

i = 1
for sde in [sde1, sde2, sde3]:
    print(f"Running SDE {i}/3: {sde.name}")
    ref_y, _ = euler_maruyama(sde, Y0, 100000, 0.001, T_END)
    em_y, em_nfe = euler_maruyama(sde, Y0, N_SAMPLES, 0.1, T_END)
    segs_y, segs_nfe = segs_exact_sampler(sde, Y0, N_SAMPLES, 0.5, T_END)
    em_w2 = np.sqrt(np.mean((np.sort(em_y) - np.sort(ref_y[:N_SAMPLES]))**2))
    segs_w2 = np.sqrt(np.mean((np.sort(segs_y) - np.sort(ref_y[:N_SAMPLES]))**2))
    i += 1
    
    results.append({
        "Topology": sde.name, 
        "EM KS": ks_2samp(em_y, ref_y).statistic, 
        "SEGS KS": ks_2samp(segs_y, ref_y).statistic,
        "EM W2": em_w2,
        "SEGS W2": segs_w2,
        "EM NFE": em_nfe, 
        "SEGS NFE": segs_nfe
    })

display_lib.display(pd.DataFrame(results))

Running SDE 1/3: Periodic
Running SDE 2/3: Sigmoidal Proxy
Running SDE 3/3: Asymmetric Periodic


,Topology,EM KS,SEGS KS,EM W2,SEGS W2,EM NFE,SEGS NFE
0,Periodic,0.01060,0.00640,0.025332,0.022530,20,8.68715
1,Sigmoidal Proxy,0.01492,0.00418,0.025826,0.008834,20,15.65185
2,Asymmetric Periodic,0.02861,0.00352,0.074907,0.020541,20,27.47545


## Time-Dependent Generative Surrogate

In [5]:
np.random.seed(42)  # For reproducibility

class TimeDependentGenerativeSDE:
    def __init__(self, gamma=0.5):
        self.gamma = gamma
        self.name = "Time-Decaying Periodic Generative SDE"

    def A(self, x, tau):
        """Non-stationary generative potential: flattens smoothly as tau -> T."""
        return np.cos(x) * np.exp(-self.gamma * tau)

    def drift(self, x, tau):
        """alpha(x, tau) = nabla_x A(x, tau)"""
        return -np.sin(x) * np.exp(-self.gamma * tau)

    def compute_slab_bounds(self, tau_start, tau_end):
        """
        Dynamically calculate strict mathematical bounds for the current time slab.
        Derivation: phi(x,tau) = 0.5*sin^2(x)*e^{-2*gamma*tau} - (gamma + 0.5)*cos(x)*e^{-gamma*tau}
        This explicitly absorbs the partial_tau A term. The global extrema of the 
        quadratic form strictly occur at cos(x) in {-1, 1} and evaluated at tau_start.
        """
        A_max = np.exp(-self.gamma * tau_end)
        phi_min = -(self.gamma + 0.5) * np.exp(-self.gamma * tau_start)
        M_intensity = 2.0 * (self.gamma + 0.5) * np.exp(-self.gamma * tau_start)
        return A_max, phi_min, M_intensity

    def phi_tilde(self, x, tau, phi_min):
        """
        Time-dependent, non-negative intensity accounting for the active partial_tau A.
        phi(x, tau) = 0.5*|alpha|^2 + 0.5*Delta_x A + partial_tau A
        """
        alpha_sq = (np.sin(x)**2) * np.exp(-2 * self.gamma * tau)
        lap_A = -np.cos(x) * np.exp(-self.gamma * tau)
        dtau_A = -self.gamma * np.cos(x) * np.exp(-self.gamma * tau)
        
        raw_phi = 0.5 * alpha_sq + 0.5 * lap_A + dtau_A
        return raw_phi - phi_min

def em_time_dependent(sde, y0, N_samples, dt, T):
    """Generates the fine-grid Reference Continuous-Time Law."""
    np.random.seed(42)
    y = np.full(N_samples, float(y0))
    n_steps = int(np.ceil(T / dt))
    nfe = 0
    tau = 0.0
    for _ in range(n_steps):
        y += sde.drift(y, tau) * dt + np.sqrt(dt) * np.random.randn(N_samples)
        tau += dt
        nfe += 1
    return y, nfe

def segs_time_dependent_sampler(sde, y0, N_samples, dtau, T):
    """SEGS adaptation tracking dynamic intensity bounds and partial_tau A."""
    np.random.seed(42)
    y = np.full(N_samples, float(y0))
    n_steps = int(np.ceil(T / dtau))
    total_nfe = 0
    
    for step in range(n_steps):
        tau_start = step * dtau
        dt_curr = min(dtau, T - tau_start)
        tau_end = tau_start + dt_curr
        
        # Fetch rigorously derived bounds localized to the current transition slab
        A_max, phi_min, M_intensity = sde.compute_slab_bounds(tau_start, tau_end)
        
        active_idx = np.arange(N_samples)
        y_next = np.zeros(N_samples)
        
        while len(active_idx) > 0:
            K_active = len(active_idx)
            y_cand = y[active_idx] + np.sqrt(dt_curr) * np.random.randn(K_active)
            total_nfe += K_active 
            
            # Endpoint acceptance utilizes the time-dependent energy A(x, tau_end)
            prob_end = np.exp(sde.A(y_cand, tau_end) - A_max)
            end_mask = np.random.rand(K_active) <= prob_end
            
            idx_pass = active_idx[end_mask]
            start_pass = y[idx_pass]
            cand_pass = y_cand[end_mask]
            K_pass = len(idx_pass)
            
            if K_pass == 0: continue
                
            K_arrivals = np.random.poisson(M_intensity * dt_curr, size=K_pass)
            path_accepted = np.ones(K_pass, dtype=bool)
            
            for j in range(K_pass):
                k_arr = K_arrivals[j]
                if k_arr == 0: continue
                    
                s_times = np.sort(np.random.uniform(0, dt_curr, k_arr))
                curr_y, curr_t = start_pass[j], 0.0
                y_end = cand_pass[j]
                reject = False
                
                for k in range(k_arr):
                    t_next = s_times[k]
                    dt_rem = dt_curr - curr_t
                    
                    mean_next = curr_y + (t_next - curr_t) / dt_rem * (y_end - curr_y)
                    var_next = (t_next - curr_t) * (dt_curr - t_next) / dt_rem
                    
                    next_y = mean_next + np.sqrt(max(var_next, 1e-12)) * np.random.randn()
                    curr_y, curr_t = next_y, t_next
                    total_nfe += 1
                    
                    # EXACT REJECTION: Evaluated at absolute generative time to test partial_tau A
                    tau_absolute = tau_start + t_next
                    intensities = sde.phi_tilde(next_y, tau_absolute, phi_min)
                    
                    if np.random.rand() <= intensities / M_intensity:
                        reject = True
                        break
                        
                if reject: path_accepted[j] = False
                    
            final_idx = idx_pass[path_accepted]
            y_next[final_idx] = cand_pass[path_accepted]
            active_idx = active_idx[~np.isin(active_idx, final_idx)]
            
        y = y_next
    return y, total_nfe / N_samples

In [6]:
# --- Validation Execution ---
print("\nRunning Time-Dependent Generative SDE Benchmark...")
time_sde = TimeDependentGenerativeSDE(gamma=0.5)
N_SAMPLES, T_END, Y0 = 20000, 2.0, 0.0

print("1. Computing Fine Ground-Truth EM Reference Proxy (dt=0.001)...")
ref_y_td, _ = em_time_dependent(time_sde, Y0, 100000, 0.001, T_END)

print("2. Computing Coarse Baseline EM (dt=0.1)...")
em_y_td, em_nfe_td = em_time_dependent(time_sde, Y0, N_SAMPLES, 0.1, T_END)

print("3. Executing Exact SEGS Sampler (dtau=0.5)...")
segs_y_td, segs_nfe_td = segs_time_dependent_sampler(time_sde, Y0, N_SAMPLES, 0.5, T_END)

em_w2_td = np.sqrt(np.mean((np.sort(em_y_td) - np.sort(ref_y_td[:N_SAMPLES]))**2))
segs_w2_td = np.sqrt(np.mean((np.sort(segs_y_td) - np.sort(ref_y_td[:N_SAMPLES]))**2))

# KS Critical Value for N=20000 against Reference N=100000 is ~0.0105
df_time = pd.DataFrame([{
    "Topology": time_sde.name, 
    "EM KS (Bias)": ks_2samp(em_y_td, ref_y_td).statistic, 
    "SEGS KS (Exact)": ks_2samp(segs_y_td, ref_y_td).statistic, 
    "EM W2 (Bias)": em_w2_td,
    "SEGS W2 (Exact)": segs_w2_td,
    "EM NFE": em_nfe_td, 
    "SEGS NFE": segs_nfe_td
}])

display_lib.display(df_time)


Running Time-Dependent Generative SDE Benchmark...
1. Computing Fine Ground-Truth EM Reference Proxy (dt=0.001)...
2. Computing Coarse Baseline EM (dt=0.1)...
3. Executing Exact SEGS Sampler (dtau=0.5)...


,Topology,EM KS (Bias),SEGS KS (Exact),EM W2 (Bias),SEGS W2 (Exact),EM NFE,SEGS NFE
0,Time-Decaying Periodic Generative SDE,0.00795,0.00505,0.022762,0.029547,20,8.66535


## Backward SDE Simulation

In [7]:
# ==============================================================================
# RIGOROUS SCORE-BASED GENERATIVE MODEL (VP-SDE à la Song et al., 2021)
# 
# Demonstrates SEGS on a true reverse-time Generative Diffusion model.
# Highlights the mathematical theorem where the Fokker-Planck identity 
# annihilates the score from the path-intensity, leaving \tilde{\phi}(y) = y^2 / 8.
# ==============================================================================

np.random.seed(42)

# --- 1. Generative Modeling Setup (VP SDE & Analytical GMM) ---
class TrueVPGenerativeSDE:
    def __init__(self, w=[0.5, 0.5], mu=[-4.0, 4.0], sigma=[0.5, 0.5], beta=1.0, T=2.0, L=15.0):
        self.w = np.array(w)
        self.mu = np.array(mu)
        self.var = np.array(sigma)**2
        self.beta = beta
        self.T = T
        self.L = L 
        self.M_intensity = (self.L ** 2) / 8.0  # \tilde{\phi}(y) = y^2 / 8
        
    def marginal_params(self, t):
        """Analytical means and variances of the GMM at forward generation time t."""
        a_t = np.exp(-self.beta * t)
        mu_t = self.mu * np.sqrt(a_t)
        var_t = self.var * a_t + 1.0 - a_t
        return mu_t, var_t
        
    def log_p(self, y, t):
        """Exact forward marginal log-density log p_t(y) (Surrogate Energy)."""
        mu_t, var_t = self.marginal_params(t)
        y = np.atleast_1d(y)[:, None]
        E_k = np.log(self.w) - 0.5 * np.log(2 * np.pi * var_t) - 0.5 * ((y - mu_t)**2) / var_t
        return logsumexp(E_k, axis=1)

    def score(self, y, t):
        """Exact analytical score \nabla_y log p_t(y) (Neural Network equivalent)."""
        mu_t, var_t = self.marginal_params(t)
        y = np.atleast_1d(y)[:, None]
        E_k = np.log(self.w) - 0.5 * np.log(2 * np.pi * var_t) - 0.5 * ((y - mu_t)**2) / var_t
        W_k = np.exp(E_k - logsumexp(E_k, axis=1, keepdims=True))
        grad_terms = -(y - mu_t) / var_t
        return np.sum(W_k * grad_terms, axis=1)

    def A_potential(self, y, tau):
        """Exact endpoint potential A(y, \tau) mapping the reverse generative flow."""
        t_fwd = max(self.T - tau / self.beta, 1e-5)
        return 0.25 * (y**2) + self.log_p(y, t_fwd)
        
    def sample_marginal(self, N, t):
        """Samples exactly from the marginal p_t(x) to isolate SDE solver bias."""
        mu_t, var_t = self.marginal_params(t)
        components = np.random.choice(len(self.w), size=N, p=self.w)
        return np.random.randn(N) * np.sqrt(var_t[components]) + mu_t[components]

# --- 2. Numerical Generative Baselines ---
def em_generative_backward(sde, yT, dt):
    """Numerically discretizes the standard reverse-time generative SDE."""
    np.random.seed(42)
    y = yT.copy()
    N = len(y)
    t_steps = np.arange(sde.T, 0, -dt)
    nfe = 0
    
    for t in t_steps:
        # Evaluate expensive network at EVERY micro-step
        drift = 0.5 * sde.beta * y + sde.beta * sde.score(y, t)
        y += drift * dt + np.sqrt(sde.beta * dt) * np.random.randn(N)
        nfe += 1
    return y, nfe

def segs_generative_sampler(sde, yT, dtau):
    """SEGS Exact Sampler utilizing the Score-Free Fokker-Planck collapse."""
    np.random.seed(42)
    y = yT.copy()
    N = len(y)
    Tau_total = sde.beta * sde.T
    n_steps = int(np.ceil(Tau_total / dtau))
    total_score_nfe = 0 
    
    for step in range(n_steps):
        tau_start = step * dtau
        dt_curr = min(dtau, Tau_total - tau_start)
        tau_end = tau_start + dt_curr
        
        # Grid-based global max extraction for robust 1D bounding
        y_grid = np.linspace(-sde.L, sde.L, 1000)
        A_max = np.max(sde.A_potential(y_grid, tau_end))
        
        active_idx = np.arange(N)
        y_next = np.zeros(N)
        
        while len(active_idx) > 0:
            K_active = len(active_idx)
            y_cand = y[active_idx] + np.sqrt(dt_curr) * np.random.randn(K_active)
            
            # --- PHASE 1: Endpoint Acceptance (Requires Network/Score Eval) ---
            valid_mask = np.abs(y_cand) <= sde.L
            prob_end = np.zeros(K_active)
            
            if np.any(valid_mask):
                A_cand = sde.A_potential(y_cand[valid_mask], tau_end)
                prob_end[valid_mask] = np.exp(A_cand - A_max)
                total_score_nfe += np.sum(valid_mask)
            
            end_mask = np.random.rand(K_active) <= prob_end
            idx_pass = active_idx[end_mask]
            start_pass, cand_pass = y[idx_pass], y_cand[end_mask]
            K_pass = len(idx_pass)
            if K_pass == 0: continue
                
            # --- PHASE 2: Poisson Thinning (ZERO Score Evaluations Required!) ---
            K_arrivals = np.random.poisson(sde.M_intensity * dt_curr, size=K_pass)
            path_accepted = np.ones(K_pass, dtype=bool)
            
            for j in range(K_pass):
                k_arr = K_arrivals[j]
                if k_arr == 0: continue
                    
                s_times = np.sort(np.random.uniform(0, dt_curr, k_arr))
                curr_y, curr_t = start_pass[j], 0.0
                y_end = cand_pass[j]
                reject = False
                
                # Sequential Brownian bridge preserving temporal covariance
                for k in range(k_arr):
                    t_next = s_times[k]
                    dt_rem = dt_curr - curr_t
                    mean_next = curr_y + (t_next - curr_t) / dt_rem * (y_end - curr_y)
                    var_next = (t_next - curr_t) * (dt_curr - t_next) / dt_rem
                    next_y = mean_next + np.sqrt(max(var_next, 1e-12)) * np.random.randn()
                    curr_y, curr_t = next_y, t_next
                    
                    if abs(next_y) > sde.L:
                        reject = True; break
                    
                    # THEORETICAL TRIUMPH: Intensity strictly evaluates y^2 / 8. No Score!
                    intensity = 0.125 * (next_y ** 2)
                    
                    if np.random.rand() <= intensity / sde.M_intensity:
                        reject = True; break
                        
                if reject: path_accepted[j] = False
                    
            final_idx = idx_pass[path_accepted]
            y_next[final_idx] = cand_pass[path_accepted]
            active_idx = active_idx[~np.isin(active_idx, final_idx)]
            
        y = y_next
    return y, total_score_nfe / N

# --- 3. Execution & Validation Protocol ---
print("Initializing Generative Score-Based SDE Benchmark (VP-SDE)...")
N_SAMPLES = 20000
vp_sde = TrueVPGenerativeSDE()

print("1. Drawing Exact Diffuse Prior States p_T(x) to isolate solver bias...")
y_prior = vp_sde.sample_marginal(N_SAMPLES, vp_sde.T)

print("2. Running Coarse Euler-Maruyama SDE Discretization (dt=0.05)...")
y_em, em_nfe = em_generative_backward(vp_sde, y_prior, dt=0.1)

print("3. Running Exact SEGS Generative Sampler (dtau=0.5)...")
y_segs, segs_nfe = segs_generative_sampler(vp_sde, y_prior, dtau=0.5)

# Ground truth data distribution at t=0
y_true = vp_sde.sample_marginal(N_SAMPLES, 0.0)

em_w2 = np.sqrt(np.mean((np.sort(y_em) - np.sort(y_true[:N_SAMPLES]))**2))
segs_w2 = np.sqrt(np.mean((np.sort(y_segs) - np.sort(y_true[:N_SAMPLES]))**2))

df_res = pd.DataFrame([{
    "Generative Target": "VP-SDE (Bimodal GMM)", 
    "EM KS (Bias)": ks_2samp(y_em, y_true).statistic, 
    "SEGS KS (Exact)": ks_2samp(y_segs, y_true).statistic, 
    "EM W2 (Bias)": em_w2,
    "SEGS W2 (Exact)": segs_w2,
    "EM NFE": em_nfe, 
    "SEGS NFE": segs_nfe
}])
display_lib.display(df_res)

Initializing Generative Score-Based SDE Benchmark (VP-SDE)...
1. Drawing Exact Diffuse Prior States p_T(x) to isolate solver bias...
2. Running Coarse Euler-Maruyama SDE Discretization (dt=0.05)...
3. Running Exact SEGS Generative Sampler (dtau=0.5)...


,Generative Target,EM KS (Bias),SEGS KS (Exact),EM W2 (Bias),SEGS W2 (Exact),EM NFE,SEGS NFE
0,VP-SDE (Bimodal GMM),0.02205,0.0059,0.166953,0.184249,20,161.87485


## High Dimension Case (16D)

In [8]:
# Ensure absolute reproducibility
np.random.seed(42)

# ==============================================================================
# 1. Rigorous High-Dimensional Metrics (Generalizing KS and W2)
# ==============================================================================
def compute_bures_wasserstein_distance(X, Y):
    """Computes the Bures-Wasserstein-2 Distance (Fréchet Pixel Distance)."""
    mu_X, sig_X = np.mean(X, axis=0), np.cov(X, rowvar=False)
    mu_Y, sig_Y = np.mean(Y, axis=0), np.cov(Y, rowvar=False)
    
    ssdiff = np.sum((mu_X - mu_Y)**2.0)
    covmean = sqrtm(sig_X.dot(sig_Y))
    if np.iscomplexobj(covmean): covmean = covmean.real
    
    return ssdiff + np.trace(sig_X + sig_Y - 2.0 * covmean)

def compute_mmd(X, Y, gamma=0.1):
    """Computes MMD, the high-dimensional generalization of the two-sample KS test."""
    XX = np.sum(X**2, axis=1)[:, None] + np.sum(X**2, axis=1)[None, :] - 2 * np.dot(X, X.T)
    YY = np.sum(Y**2, axis=1)[:, None] + np.sum(Y**2, axis=1)[None, :] - 2 * np.dot(Y, Y.T)
    XY = np.sum(X**2, axis=1)[:, None] + np.sum(Y**2, axis=1)[None, :] - 2 * np.dot(X, Y.T)
    
    return np.exp(-gamma * XX).mean() + np.exp(-gamma * YY).mean() - 2 * np.exp(-gamma * XY).mean()

In [9]:
# ==============================================================================
# 2. Factorizable 16-D Target Proxy (Stiff SDE)
# ==============================================================================
D_DIM = 16
C = 0.25  # Sharpened parameter creates stiff gradients

target_pattern = np.array([
    1, -1, 1, -1,
   -1, 1, -1, 1,
    1, -1, 1, -1,
   -1, 1, -1, 1
], dtype=np.float32)

def A_1d(x, tgt): return -np.sqrt((x - tgt)**2 + C**2)
def drift_1d(x, tgt): return -(x - tgt) / np.sqrt((x - tgt)**2 + C**2)
def lap_1d(x, tgt): return -C**2 / ((x - tgt)**2 + C**2)**1.5
def phi_1d(x, tgt): return 0.5 * drift_1d(x, tgt)**2 + 0.5 * lap_1d(x, tgt) + 1.0 / (2 * C)

A_MAX_1D = -C
M_INT_1D = 0.5 + 1.0 / (2 * C) 

In [10]:
# ==============================================================================
# 3. Solvers
# ==============================================================================
def euler_maruyama(y_noisy, dt, T):
    np.random.seed(42)
    y = y_noisy.copy()
    N_samples = y.shape[0]
    n_steps = int(T / dt)
    nfe = 0
    for _ in range(n_steps):
        diff = y - target_pattern
        drift = -diff / np.sqrt(diff**2 + C**2)
        y += drift * dt + np.sqrt(dt) * np.random.randn(N_samples, D_DIM)
        nfe += 1
    return y, nfe

def segs_16d_sampler(y_noisy, dtau, T):
    np.random.seed(42)
    # SPATIAL FACTORIZATION
    y_flat = y_noisy.copy().flatten()
    N_samples = y_noisy.shape[0]
    tgt_flat = np.tile(target_pattern, N_samples)
    N_total = len(y_flat)
    
    n_steps = int(np.ceil(T / dtau))
    total_scalar_nfe = 0
    
    for step in range(n_steps):
        dt_curr = min(dtau, T - step * dtau)
        active_idx = np.arange(N_total)
        y_next_flat = np.zeros(N_total)
        
        while len(active_idx) > 0:
            K_active = len(active_idx)
            y_cand = y_flat[active_idx] + np.sqrt(dt_curr) * np.random.randn(K_active)
            total_scalar_nfe += K_active
            
            prob_end = np.exp(A_1d(y_cand, tgt_flat[active_idx]) - A_MAX_1D)
            end_mask = np.random.rand(K_active) <= prob_end
            
            idx_pass = active_idx[end_mask]
            start_pass, cand_pass = y_flat[idx_pass], y_cand[end_mask]
            tgt_pass = tgt_flat[idx_pass]
            K_pass = len(idx_pass)
            
            if K_pass == 0: continue
            
            K_arrivals = np.random.poisson(M_INT_1D * dt_curr, size=K_pass)
            accepted = np.ones(K_pass, dtype=bool)
            
            for j in range(K_pass):
                k_arr = K_arrivals[j]
                if k_arr == 0: continue
                
                # Ensure sequential simulation of Brownian Bridge to preserve temporal correlation
                s_times = np.sort(np.random.uniform(0, dt_curr, k_arr))
                curr_y, curr_t = start_pass[j], 0.0
                y_end = cand_pass[j]
                reject = False
                
                for k in range(k_arr):
                    t_next = s_times[k]
                    dt_rem = dt_curr - curr_t
                    
                    mean_next = curr_y + (t_next - curr_t) / dt_rem * (y_end - curr_y)
                    var_next = (t_next - curr_t) * (dt_curr - t_next) / dt_rem
                    
                    next_y = mean_next + np.sqrt(max(var_next, 1e-12)) * np.random.randn()
                    curr_y, curr_t = next_y, t_next
                    total_scalar_nfe += 1
                    
                    intensity = phi_1d(next_y, tgt_pass[j])
                    if np.random.rand() <= intensity / M_INT_1D:
                        reject = True
                        break
                        
                if reject: accepted[j] = False
            
            final_idx = idx_pass[accepted]
            y_next_flat[final_idx] = cand_pass[accepted]
            active_idx = active_idx[~np.isin(active_idx, final_idx)]
            
        y_flat = y_next_flat
    
    vector_nfe = (total_scalar_nfe / D_DIM) / N_samples
    return y_flat.reshape(N_samples, D_DIM), vector_nfe

In [11]:
# ==============================================================================
# 4. Execution Pipeline
# ==============================================================================

N_SAMPLES = 2000
N_REF = 10000  
T_END = 1.0

print(f"--- Running Stiff 16D Spatial-Factorized Simulation on CPU ---")
y_noisy_ref = np.random.randn(N_REF, D_DIM) * 2.0 
y_noisy = y_noisy_ref[:N_SAMPLES]

print("1. Generating Continuous-Time Ground Truth (Ultra-fine EM, dt=0.002)...")
y_ref, _ = euler_maruyama(y_noisy_ref, dt=0.002, T=T_END)

print("\n2. Running Coarse Baseline EM (dt=0.1)...")
y_em, em_nfe = euler_maruyama(y_noisy, dt=0.1, T=T_END)

print("\n3. Running Exact SEGS Sampler (dtau=0.5)...")
y_segs, segs_nfe = segs_16d_sampler(y_noisy, dtau=0.5, T=T_END)

mmd_em = compute_mmd(y_ref[:N_SAMPLES], y_em)
mmd_segs = compute_mmd(y_ref[:N_SAMPLES], y_segs)

w2_em = compute_bures_wasserstein_distance(y_ref[:N_SAMPLES], y_em)
w2_segs = compute_bures_wasserstein_distance(y_ref[:N_SAMPLES], y_segs)

--- Running Stiff 16D Spatial-Factorized Simulation on CPU ---
1. Generating Continuous-Time Ground Truth (Ultra-fine EM, dt=0.002)...

2. Running Coarse Baseline EM (dt=0.1)...

3. Running Exact SEGS Sampler (dtau=0.5)...


In [12]:
results = []
ems = [w2_em, mmd_em, em_nfe]
segs = [w2_segs, mmd_segs, segs_nfe]

i = 0
for name in ["Bures-Wasserstein W2 (FPD)", "Max Mean Discrepancy (KS Proxy)", "NFE"]:
    results.append({
        "Metric": name, 
        "EM": ems[i], 
        "SEGS": segs[i],
    })
    i += 1

display_lib.display(pd.DataFrame(results))

,Metric,EM,SEGS
0,Bures-Wasserstein W2 (FPD),0.806493,0.103069
1,Max Mean Discrepancy (KS Proxy),0.001468,0.000840
2,NFE,10.000000,95.794594
